# Healthcare Claims EDA & Cost Analysis
**CMS DE-SynPUF Medicare Claims | 2008-2010 | 230M+ Records**

---

**Objective:** Answer six core business questions a US health plan asks every quarter - where the money goes, for whom, and what's avoidable - using 230 million synthetic Medicare claims processed with DuckDB.

| # | Business Question |
|---|---|
| 1 | What share of total spend comes from the costliest members? |
| 2 | What is the PMPM cost by service line, and how does it trend? |
| 3 | How does chronic-condition layering drive cost? |
| 4 | What is the 30-day readmission rate, and what drives it? |
| 5 | How does cost vary by age and sex? |
| 6 | What are the top cost drivers by DRG, HCPCS, and drug category? |

## 0 - Setup

In [103]:
import sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# src/ on path
sys.path.insert(0, str(Path.cwd().parent))

from src.config          import *
from src.data_loader     import get_connection, create_views, create_beneficiary_enriched, row_count
from src.quality         import compute_completeness, print_completeness_tiers, check_referential_integrity, check_temporal_consistency
from src.visual_identity import (
    visual_identity, fig_title, adjust_axis, auto_scale_axis,
    fmt_num, fmt_usd, bar_labels, render_table,
    ECON_BG, ECON_RED, NATURE_TEAL, NATURE_CORAL, DEEP_BLUE,
    AMBER, PLUM, WARM_GREY, CHARCOAL, LIGHT_GREY,
    PALETA_SEQ, WHITE
)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# DuckDB connection & views─
con = get_connection(memory_limit="6GB", threads=4)
create_views(con)

print("✓ Views created for:", list(DESYNPUF_PATHS.keys()))
print(f"✓ DuckDB ready  ·  memory_limit = 6 GB  ·  threads = 4")

✓ Views created for: ['beneficiary', 'inpatient', 'outpatient', 'carrier', 'pde']
✓ DuckDB ready  ·  memory_limit = 6 GB  ·  threads = 4


In [ ]:
# Artifact store and DuckDB resource settings
# The 6 GB limit set in 0 stays the notebook default; heavy cells raise it only
# inside `with duckdb_settings(con, **HEAVY):` and it is restored on exit.
import os
from src.artifacts import ArtifactStore, duckdb_settings

ARTIFACTS_DIR = Path.cwd().parent / "artifacts"
SPILL_DIR     = ARTIFACTS_DIR / "_duckdb_spill"
SPILL_DIR.mkdir(parents=True, exist_ok=True)

# 70% of physical RAM (Linux sysconf); never below the 6 GB already in use
TOTAL_RAM_GB = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1024**3
HEAVY_MEM_GB = max(6, int(TOTAL_RAM_GB * 0.70))

# temp_directory is set once for the whole session, before any spill happens,
# and is NOT part of HEAVY: DuckDB refuses to switch it after spill files were
# written there ("Cannot switch temporary directory after the current one has
# been used"), so a per-cell set/restore cannot work.
_spill_literal = str(SPILL_DIR).replace("'", "''")
try:
    con.execute(f"SET temp_directory = '{_spill_literal}'")
except Exception as exc:
    print(f"[duckdb] temp_directory not changed (already used in this session): {exc}")
_current_tmp = con.execute("SELECT current_setting('temp_directory')").fetchone()[0]
print(f"[duckdb] temp_directory = {_current_tmp}")

HEAVY = dict(
    memory_limit=f"{HEAVY_MEM_GB}GB",
    threads=2,
    preserve_insertion_order=False,
)
print(f"Physical RAM: {TOTAL_RAM_GB:,.1f} GB | heavy memory_limit: {HEAVY_MEM_GB} GB | spill: {SPILL_DIR}")

store = ArtifactStore(con, ARTIFACTS_DIR)


def _dt(col):
    # Same BIGINT YYYYMMDD -> DATE parsing used elsewhere in the notebook
    return (f"COALESCE(TRY_CAST({col} AS DATE), "
            f"TRY_STRPTIME(CAST({col} AS VARCHAR), '%Y%m%d')::DATE)")


def _show_df(title, df):
    # Single-row results print one metric per line; multi-row results as a table
    print(f"\n  [{title}]")
    if len(df) == 1:
        for k, v in df.iloc[0].items():
            if isinstance(v, (int, float, np.integer, np.floating)) and pd.notna(v):
                print(f"    {k:<42s} {v:>16,.0f}")
            else:
                print(f"    {k:<42s} {str(v):>16s}")
    else:
        print(df.to_string(index=False))

[duckdb] temp_directory = /home/aosousa/Documents/PORTFOLIO/artifacts/_duckdb_spill
Physical RAM: 19.4 GB | heavy memory_limit: 13 GB | spill: /home/aosousa/Documents/PORTFOLIO/artifacts/_duckdb_spill
[artifacts] store at /home/aosousa/Documents/PORTFOLIO/artifacts | 47 artifact(s) in manifest


## 1 - Data Quality Assessment

Before any analysis, we validate the data across three dimensions:
1. **Volume** - do row counts match expectations?
2. **Completeness** - what's the fill rate per column?
3. **Referential integrity** - do foreign keys resolve?
4. **Temporal consistency** - are date ranges plausible?

In [32]:
# 1a. Volume check
print("TABLE VOLUME CHECK")
print("=" * 60)
for table, expected in EXPECTED_ROWS.items():
    actual = row_count(con, table)
    status = "✓" if actual == expected else "✗"
    delta  = actual - expected
    print(f"  {status} {table:<14s}  {actual:>14,}  (expected {expected:>14,}, Δ {delta:+,})")

TABLE VOLUME CHECK
  ✓ beneficiary          6,873,274  (expected      6,873,274, Δ +0)
  ✓ carrier             94,863,452  (expected     94,863,452, Δ +0)
  ✓ inpatient            1,332,822  (expected      1,332,822, Δ +0)
  ✓ outpatient          15,826,987  (expected     15,826,987, Δ +0)
  ✓ pde                111,085,969  (expected    111,085,969, Δ +0)


In [33]:
# 1b. Completeness by table
for table in ["beneficiary", "inpatient", "outpatient", "carrier", "pde"]:
    print(f"\n{'─'*60}")
    print(f"COMPLETENESS: {table.upper()}")
    print(f"{'─'*60}")
    df_c = compute_completeness(con, table, exclude_prefixes=[])
    print_completeness_tiers(df_c, max_per_tier=20)


────────────────────────────────────────────────────────────
COMPLETENESS: BENEFICIARY
────────────────────────────────────────────────────────────
Completeness summary (34 columns):

  100% complete: 33 columns
    DESYNPUF_ID                              100.00%
    BENE_BIRTH_DT                            100.00%
    BENE_SEX_IDENT_CD                        100.00%
    BENE_RACE_CD                             100.00%
    BENE_ESRD_IND                            100.00%
    SP_STATE_CODE                            100.00%
    BENE_COUNTY_CD                           100.00%
    BENE_HI_CVRAGE_TOT_MONS                  100.00%
    BENE_SMI_CVRAGE_TOT_MONS                 100.00%
    BENE_HMO_CVRAGE_TOT_MONS                 100.00%
    PLAN_CVRG_MOS_NUM                        100.00%
    SP_ALZHDMTA                              100.00%
    SP_CHF                                   100.00%
    SP_CHRNKIDN                              100.00%
    SP_CNCR                                  

In [34]:
# 1c. Referential integrity
print("REFERENTIAL INTEGRITY CHECKS")
print("=" * 60)

ri_checks = [
    ("inpatient",  "beneficiary", "DESYNPUF_ID", "DESYNPUF_ID", "Inpatient → Beneficiary"),
    ("outpatient", "beneficiary", "DESYNPUF_ID", "DESYNPUF_ID", "Outpatient → Beneficiary"),
    ("carrier",    "beneficiary", "DESYNPUF_ID", "DESYNPUF_ID", "Carrier → Beneficiary"),
    ("pde",        "beneficiary", "DESYNPUF_ID", "DESYNPUF_ID", "PDE → Beneficiary"),
]

for left, right, lk, rk, desc in ri_checks:
    check_referential_integrity(con, left, right, lk, rk, desc)

REFERENTIAL INTEGRITY CHECKS


In [35]:
# 1d. Temporal consistency
print("TEMPORAL CONSISTENCY CHECKS")
print("=" * 60)

tc_checks = [
    ("inpatient",  "CLM_FROM_DT",  "CLM_THRU_DT",  "2008-01-01", "2010-12-31"),
    ("outpatient", "CLM_FROM_DT",  "CLM_THRU_DT",  "2008-01-01", "2010-12-31"),
    ("carrier",    "CLM_FROM_DT",  "CLM_THRU_DT",  "2008-01-01", "2010-12-31"),
    ("pde",        "SRVC_DT",       None,           "2008-01-01", "2010-12-31"),
]

for table, date_from, date_thru, exp_min, exp_max in tc_checks:
    if date_thru:
        check_temporal_consistency(con, table, date_from, date_thru, exp_min, exp_max)
    else:
        # PDE has only a single service date (SRVC_DT).
        # DE-SynPUF stores dates as YYYYMMDD integers (BIGINT), so we
        # cast to VARCHAR then parse to DATE before comparing to date literals.
        date_cast = (
            f"COALESCE(TRY_CAST({date_from} AS DATE), "
            f"TRY_STRPTIME(CAST({date_from} AS VARCHAR), '%Y%m%d')::DATE)"
        )
        result = con.execute(f"""
            SELECT
                MIN({date_cast})::VARCHAR AS min_dt,
                MAX({date_cast})::VARCHAR AS max_dt,
                SUM(CASE WHEN {date_cast} < DATE '{exp_min}'
                           OR {date_cast} > DATE '{exp_max}'
                         THEN 1 ELSE 0 END) AS out_of_range
            FROM {table}
            WHERE {date_from} IS NOT NULL
        """).fetchdf()
        r = result.iloc[0]
        oor = int(r["out_of_range"])
        status = "✓" if oor == 0 else f"✗ ({oor:,} out of range)"
        print(f"  {status}  {table}.{date_from}: [{r['min_dt']} ... {r['max_dt']}]")

TEMPORAL CONSISTENCY CHECKS


  ✓  pde.SRVC_DT: [2008-01-01 ... 2010-12-31]


> **Data quality verdict:** The DE-SynPUF dataset is synthetic by design - CMS generated it for research and education.<br>
We expect full referential integrity across all joins and dates within the 2008-2010 window.<br>
Completeness gaps in optional fields (e.g., diagnosis codes 4-10, secondary procedures) reflect clinical reality.<br>
Not every claim carries the maximum number of codes.

In [105]:
# Typed claim bases for INPATIENT and OUTPATIENT
# Dates are parsed once, CLM_ID/PRVDR_NUM are cast to VARCHAR, and every row gets
# claim_key = CLM_ID || '-' || SEGMENT.
# CLM_ID can carry two rows with different PRVDR_NUM, so CLM_ID alone is not a claim identifier.
# (CLM_ID, SEGMENT) is the atomic record. Uniqueness is asserted bELOW.
# Only the columns needed for family netting, stays and DRG ranking are kept.

CLM_IP_SQL = f"""
    SELECT
        CAST(CLM_ID AS VARCHAR) || '-' || CAST(SEGMENT AS VARCHAR) AS claim_key,
        CAST(CLM_ID AS VARCHAR)            AS CLM_ID,
        CAST(SEGMENT AS INTEGER)           AS SEGMENT,
        DESYNPUF_ID,
        CAST(PRVDR_NUM AS VARCHAR)         AS PRVDR_NUM,
        {_dt('CLM_ADMSN_DT')}              AS adm_dt,
        {_dt('CLM_FROM_DT')}               AS from_dt,
        {_dt('CLM_THRU_DT')}               AS thru_dt,
        {_dt('NCH_BENE_DSCHRG_DT')}        AS dschrg_dt,
        CLM_PMT_AMT,
        NCH_PRMRY_PYR_CLM_PD_AMT,
        CAST(CLM_DRG_CD AS VARCHAR)        AS CLM_DRG_CD
    FROM inpatient
"""

CLM_OP_SQL = f"""
    SELECT
        CAST(CLM_ID AS VARCHAR) || '-' || CAST(SEGMENT AS VARCHAR) AS claim_key,
        CAST(CLM_ID AS VARCHAR)            AS CLM_ID,
        CAST(SEGMENT AS INTEGER)           AS SEGMENT,
        DESYNPUF_ID,
        CAST(PRVDR_NUM AS VARCHAR)         AS PRVDR_NUM,
        {_dt('CLM_FROM_DT')}               AS from_dt,
        {_dt('CLM_THRU_DT')}               AS thru_dt,
        CLM_PMT_AMT,
        NCH_PRMRY_PYR_CLM_PD_AMT
    FROM outpatient
"""

with duckdb_settings(con, **HEAVY):
    store.materialize("clm_ip", CLM_IP_SQL, inputs=f"{DESYNPUF_PATHS['inpatient']}/*.csv") # materialize as csv for further inspection
    store.materialize("clm_op", CLM_OP_SQL, inputs=f"{DESYNPUF_PATHS['outpatient']}/*.csv") # materialize as csv for further inspection

    # Validation assumptions
    # 1) row count must equal the source;
    # 2) claim_key must be unique; and
    # 3) payment total must equal the source total (no row lost or added)
    for name, src_tbl in [("clm_ip", "inpatient"), ("clm_op", "outpatient")]:
        v = con.execute(f"""
            SELECT COUNT(*)                                  AS n_rows,
                   COUNT(DISTINCT claim_key)                 AS n_claim_key,
                   COUNT(*) FILTER (WHERE from_dt IS NULL)   AS from_dt_null,
                   COUNT(*) FILTER (WHERE thru_dt IS NULL)   AS thru_dt_null,
                   SUM(CLM_PMT_AMT)                          AS sum_pmt
            FROM {name}
        """).fetchdf().iloc[0]
        src_sum = con.execute(f"SELECT SUM(CLM_PMT_AMT) FROM {src_tbl}").fetchone()[0]
        print(f"\n  VALIDATION {name}")
        print(f"    rows                {int(v['n_rows']):>16,}  (expected {EXPECTED_ROWS[src_tbl]:,})")
        print(f"    distinct claim_key  {int(v['n_claim_key']):>16,}")
        print(f"    from_dt NULL        {int(v['from_dt_null']):>16,}")
        print(f"    thru_dt NULL        {int(v['thru_dt_null']):>16,}")
        print(f"    sum CLM_PMT_AMT     {v['sum_pmt']:>16,.2f}  (source {src_sum:,.2f})")
        assert int(v["n_rows"]) == EXPECTED_ROWS[src_tbl], f"{name}: row count differs from source"
        assert int(v["n_rows"]) == int(v["n_claim_key"]), f"{name}: claim_key is not unique"
        assert abs(float(v["sum_pmt"]) - float(src_sum)) < 0.005, f"{name}: payment total differs from source"
    print("\n  ✓ typed bases validated")

[duckdb] SET memory_limit = '13GB'  (was '5.5 GiB')
[duckdb] SET threads = 2  (was 4)
[duckdb] SET preserve_insertion_order = False  (was True)
[artifacts] HIT   clm_ip                                   rows=     1,332,822  built=2026-11-18T07:52:18  key=dbb3dffbbb92
[artifacts] HIT   clm_op                                   rows=    15,826,987  built=2026-11-18T07:53:24  key=a77cb164598e

  VALIDATION clm_ip
    rows                       1,332,822  (expected 1,332,822)
    distinct claim_key         1,332,822
    from_dt NULL                   1,289
    thru_dt NULL                   1,289
    sum CLM_PMT_AMT     12,710,862,270.00  (source 12,710,862,270.00)

  VALIDATION clm_op
    rows                      15,826,987  (expected 15,826,987)
    distinct claim_key        15,826,987
    from_dt NULL                 222,803
    thru_dt NULL                 222,803
    sum CLM_PMT_AMT     4,482,288,370.00  (source 4,482,288,370.00)

  ✓ typed bases validated
[duckdb] restored memory_lim